# herder - fine-tuning the merge model

The merge step asks an NLI cross-encoder whether a new claim duplicates, refines, replaces or is unrelated to a stored one. Six off-the-shelf checkpoints were measured on `bench/nli_pairs.json` and **none is good at both halves** (`NOTES.md`):

| Failure | Who gets it right | Who gets it wrong |
| --- | --- | --- |
| a correction read as neutral against the claim it replaces | general SNLI/MNLI models | the VitaminC models |
| two compatible sentences read as a contradiction | the VitaminC models | general SNLI/MNLI models |
| an open item about a claim read as replacing it | nobody | everybody |

So this trains one model for all three, from **`cross-encoder/nli-deberta-v3-base`** - the model currently running, Apache-2.0, so the licence lineage stays clean.

**Licence, and it is a real decision.** With VitaminC in, the training data is CC BY-SA 3.0. With `--no-share-alike` it is WANLI (CC BY 4.0) plus the hand-written pairs, which fixes the false contradiction and does nothing for `supersede`. **ANLI is excluded either way** - CC BY-NC would put a non-commercial claim on the weights.

**Runtime:** Colab, `Runtime -> Change runtime type -> T4 GPU`. About an hour for two epochs. Nothing here needs an API key.

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4")
!pip -q install "transformers>=4.44" datasets sentence-transformers accelerate

## 1. The repository, for the build script and the labelled pairs

The dataset is built here rather than uploaded, so the recipe in the repository is the only definition of it. `build_dataset.py` streams VitaminC and WANLI from Hugging Face, pairs VitaminC **claim against claim** by case (the shape herder actually compares), adds the hand-written open-item pairs, and drops anything matching a benchmark sentence.

In [ ]:
REPO_URL = "https://github.com/MoAbboud/PortFolio.git"
SHARE_ALIKE = True  # False -> WANLI and the hand-written pairs only

import os, pathlib

if not pathlib.Path("PortFolio").exists():
    !git clone --depth 1 $REPO_URL
os.chdir("/content/PortFolio/herder")
print(pathlib.Path.cwd())

flags = "" if SHARE_ALIKE else "--no-share-alike"
!python -m training.nli.build_dataset --out training/nli/data $flags

In [ ]:
import json, pathlib, collections

def read(name):
    return [json.loads(line) for line in pathlib.Path(f"training/nli/data/{name}.jsonl").read_text(encoding="utf-8").splitlines()]

train_rows, dev_rows = read("train"), read("dev")
print(f"{len(train_rows):,} train / {len(dev_rows):,} dev")
print(collections.Counter(r["label"] for r in train_rows))
print(collections.Counter(r["source"] for r in train_rows))
for row in train_rows[:3]:
    print(f"\n[{row['shape']} -> {row['label']}]\n  a: {row['a'][:110]}\n  b: {row['b'][:110]}")

## 2. Labels

**The base model's own label order is kept.** herder reads `id2label` from the checkpoint at inference and refuses a model it cannot interpret (`herder/core/nli.py`); training with a different order would produce a model that silently inverts every merge verdict.

In [ ]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification

BASE = "cross-encoder/nli-deberta-v3-base"
config = AutoConfig.from_pretrained(BASE)
id2label = {int(k): str(v).lower() for k, v in config.id2label.items()}
label2id = {v: k for k, v in id2label.items()}
print("the base model's order, which we keep:", id2label)
assert set(id2label.values()) == {"contradiction", "entailment", "neutral"}, id2label

In [ ]:
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(BASE)
MAX_LEN = 128  # herder compares single claims; the tail beyond this is rare and cheap to lose

def encode(rows):
    ds = Dataset.from_list([{"a": r["a"], "b": r["b"], "labels": label2id[r["label"]]} for r in rows])
    return ds.map(
        lambda batch: tokenizer(batch["a"], batch["b"], truncation=True, max_length=MAX_LEN),
        batched=True,
        remove_columns=["a", "b"],
    )

train_ds, dev_ds = encode(train_rows), encode(dev_rows)
print(train_ds)

## 3. Train

Two epochs at 2e-5. The run is checkpointed to Drive-free local storage; if the session dies, re-run from the top - the dataset build is deterministic given the seed in `recipe.json`.

In [ ]:
import inspect

import numpy as np
import transformers
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

print("transformers", transformers.__version__)
OUT = "/content/nli-herder"
EPOCHS, BATCH = 2, 32
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=3, id2label=id2label, label2id=label2id)

def metrics(pred):
    predicted = pred.predictions.argmax(-1)
    gold = pred.label_ids
    out = {"accuracy": float((predicted == gold).mean())}
    for name, index in label2id.items():
        mask = gold == index
        if mask.any():
            out[f"recall_{name}"] = float((predicted[mask] == index).mean())
    return out

# TrainingArguments changed between transformers 4 and 5: `warmup_ratio` is gone and
# `evaluation_strategy` was renamed `eval_strategy`. Rather than pin a version that will age,
# the wanted settings are translated to whatever this install accepts, and anything genuinely
# unsupported is printed rather than dropped silently - a warmup that vanished without a word
# would change the run and leave no trace of why.
wanted = dict(
    output_dir=OUT,
    num_train_epochs=EPOCHS,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=64,
    warmup_ratio=0.06,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=200,
    report_to=[],
)
supported = set(inspect.signature(TrainingArguments.__init__).parameters)

if "warmup_ratio" not in supported and "warmup_steps" in supported:
    ratio = wanted.pop("warmup_ratio")
    steps_per_epoch = max(1, len(train_ds) // BATCH)
    wanted["warmup_steps"] = int(ratio * steps_per_epoch * EPOCHS)
    print(f"warmup_ratio {ratio} -> warmup_steps {wanted['warmup_steps']}")
if "eval_strategy" not in supported and "evaluation_strategy" in supported:
    wanted["evaluation_strategy"] = wanted.pop("eval_strategy")

dropped = {k: wanted.pop(k) for k in list(wanted) if k not in supported}
if dropped:
    print("not supported by this transformers version, dropped:", dropped)

trainer = Trainer(
    model=model,
    args=TrainingArguments(**wanted),
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrics,
)
trainer.train()

In [ ]:
FINAL = "/content/nli-herder-final"
trainer.save_model(FINAL)
tokenizer.save_pretrained(FINAL)
print(trainer.evaluate())

## 4. The test that decides it

Dev accuracy is not the question. The question is what herder's **merge verdicts** become, so the model is scored on `bench/nli_pairs.json` through the real rules - the reversal path first when a candidate announces a change, then `decide_against`.

**Only the 20 `written` pairs decide.** The 10 `observed` pairs are the failures that prompted this work, so choosing on them would be selecting on the test set; they are printed separately. The real confirmation is a benchmark run back on the laptop.

In [ ]:
import json, math, pathlib, collections
from sentence_transformers import CrossEncoder
from herder.domain.merge import Label, Match, announces_change, claim_of, decide_against, decide_reversal

FLOOR = 0.5
pairs = json.loads(pathlib.Path("bench/nli_pairs.json").read_text(encoding="utf-8"))["pairs"]

def scorer(path):
    encoder = CrossEncoder(path)
    order = {int(k): str(v).lower() for k, v in encoder.config.id2label.items()}
    def both_ways(a, b):
        raw = encoder.predict([(a, b), (b, a)])
        out = []
        for row in raw:
            values = [math.exp(float(x)) for x in row]
            total = sum(values)
            probs = [v / total for v in values]
            best = max(range(len(probs)), key=probs.__getitem__)
            out.append(Label(order[best], probs[best]))
        return out[0], out[1]
    return both_ways

def verdicts(both_ways):
    right = collections.Counter(); total = collections.Counter(); wrong = []
    for pair in pairs:
        match = Match(entry_id=None, status="active", title="", text=pair["existing"], similarity=1.0)
        got = None
        if announces_change(pair["candidate"]):
            readings = [both_ways(pair["candidate"], pair["existing"])]
            claims = (claim_of(pair["candidate"]), claim_of(pair["existing"]))
            if claims != (pair["candidate"].strip(), pair["existing"].strip()):
                readings.append(both_ways(*claims))
            reversal = decide_reversal(match, readings)
            got = reversal.verdict if reversal else None
        if got is None:
            forward, backward = both_ways(pair["candidate"], pair["existing"])
            got = decide_against(match, forward, backward, FLOOR).verdict
        total[pair["group"]] += 1
        if str(got) == pair["expect"]:
            right[pair["group"]] += 1
        else:
            wrong.append((pair, str(got)))
    return right, total, wrong

for name, path in (("before (base)", BASE), ("after (fine-tuned)", FINAL)):
    right, total, wrong = verdicts(scorer(path))
    print(f"{name:22} written {right['written']}/{total['written']}   observed {right['observed']}/{total['observed']}")
    if name.startswith("after"):
        for pair, got in wrong:
            print(f"    [{pair['group']}/{pair['shape']}] expected {pair['expect']}, got {got}")

## 5. Take the weights home

Download the zip, unpack it into `herder/models/nli-herder-v1/` (that directory is gitignored - weights are rebuildable from this notebook, the recipe is what belongs in git), and point the stack at it:

```powershell
$env:HERDER_NLI_MODEL = "/models/nli-herder-v1"   # inside the container; ./models is mounted there
docker compose up -d --no-deps worker
python -m bench.nli_compare --models ./models/nli-herder-v1
```

Then the real test, against the current reference run:

```powershell
python -m bench.run --label nli-trained --methods herder
```

Watch the **supersedes in the database**, not recall: the reader disagrees with itself on about 3.4% of verdicts, so anything under ~4 facts is noise.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/nli-herder-v1", "zip", FINAL)
files.download("/content/nli-herder-v1.zip")